## Import


In [ ]:
import os
import sep
import cv2
import glob
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
from datetime import datetime
import matplotlib.dates as mdates
import matplotlib.cm as cm
import astroalign as aa
from pandas import read_csv
from ultralytics import YOLO
from pathlib import Path
import shutil
import random


## 自定义函数


In [ ]:
def _gaussian_psf(radius_px: float, size: int = None) -> np.ndarray:
    sigma = max(0.2, float(radius_px))
    if size is None:
        size = int(np.ceil(6 * sigma)) | 1
    c = size // 2
    y, x = np.mgrid[-c:c+1, -c:c+1]
    psf = np.exp(-(x*x + y*y) / (2 * sigma * sigma))
    psf /= psf.sum()
    return psf.astype(np.float32)

def _motion_kernel(length_px: float, angle_rad: float, width: float = 1.0) -> np.ndarray:
    L = max(1.0, float(length_px))
    size = int(np.ceil(L)) | 1
    c = size // 2
    y, x = np.mgrid[-c:c+1, -c:c+1]
    ca, sa = np.cos(angle_rad), np.sin(angle_rad)
    u =  ca * x + sa * y
    v = -sa * x + ca * y
    line = (np.abs(u) <= (L / 2)).astype(np.float32)
    blur = np.exp(-(v * v) / (2 * (max(0.3, width) ** 2)))
    k = line * blur
    k /= k.sum()
    return k.astype(np.float32)

def _fft_convolve_full(a: np.ndarray, b: np.ndarray) -> np.ndarray:
    """full convolution (size = (Ha+Hb-1, Wa+Wb-1)) via FFT"""
    Ha, Wa = a.shape
    Hb, Wb = b.shape
    H, W = Ha + Hb - 1, Wa + Wb - 1
    fa = np.fft.rfft2(a, s=(H, W))
    fb = np.fft.rfft2(b, s=(H, W))
    out = np.fft.irfft2(fa * fb, s=(H, W))
    return out.astype(np.float32)

def _place_kernel_add(img: np.ndarray, ker: np.ndarray, y: float, x: float, gain: float) -> None:
    """越界直接丢弃（不环绕）。"""
    H, W = img.shape
    kh, kw = ker.shape
    cy, cx = kh // 2, kw // 2

    y0 = int(np.floor(y))
    x0 = int(np.floor(x))
    dy = y - y0
    dx = x - x0

    weights = [
        (y0,   x0,   (1-dy)*(1-dx)),
        (y0+1, x0,   dy*(1-dx)),
        (y0,   x0+1, (1-dy)*dx),
        (y0+1, x0+1, dy*dx),
    ]

    for yy, xx, w in weights:
        top = yy - cy
        left = xx - cx

        r0 = max(0, top)
        c0 = max(0, left)
        r1 = min(H, top + kh)
        c1 = min(W, left + kw)

        if r0 >= r1 or c0 >= c1:
            continue

        kr0 = r0 - top
        kc0 = c0 - left
        kr1 = kr0 + (r1 - r0)
        kc1 = kc0 + (c1 - c0)

        img[r0:r1, c0:c1] += (gain * w) * ker[kr0:kr1, kc0:kc1]

def simulate_trajectory_projection_with_interval(
    seed: int,
    img_shape=(256, 256),
    radius_px=1.5,
    peak=2000.0,
    start_xy=None,              # (x, y)
    speed_px_s=8.0,
    angle_deg=None,
    n_frames=50,
    exposure_s=0.01,
    frame_interval_s=0.04,
    jitter_px=0.5,
    scintillation=0.15,
    background=50.0,
    shot_noise=False,
    read_noise=3.0,
    periodic_amp_px=None,
    periodic_along_amp_px=None,
    period_frames=None,
    return_time_proj=False,
):
    rng = np.random.default_rng(seed)
    H, W = img_shape

    if angle_deg is None:
        angle_deg = rng.uniform(0, 90)
    angle = np.deg2rad(float(angle_deg))
    dirx, diry = np.cos(angle), np.sin(angle)
    vx = float(speed_px_s) * dirx
    vy = float(speed_px_s) * diry

    if start_xy is None:
        x0 = rng.uniform(W // 4, W // 4 * 3)
        y0 = rng.uniform(H // 4, H // 4 * 3)
    else:
        x0, y0 = map(float, start_xy)

    n_frames_i = max(int(n_frames), 1)
    if period_frames is None:
        period_frames = rng.uniform(max(6.0, n_frames_i * 0.3), max(10.0, n_frames_i * 0.9))
    if periodic_amp_px is None:
        periodic_amp_px = rng.uniform(0.3, 2.2)
    if periodic_along_amp_px is None:
        periodic_along_amp_px = rng.uniform(0.0, 1.2)
    phase_perp = rng.uniform(0.0, 2.0 * np.pi)
    phase_along = rng.uniform(0.0, 2.0 * np.pi)

    psf = _gaussian_psf(radius_px)
    blur_len = max(0.0, float(speed_px_s) * float(exposure_s))
    if blur_len >= 1.0:
        mk = _motion_kernel(blur_len, angle, width=max(0.8, radius_px * 0.6))
        eff = _fft_convolve_full(psf, mk)
        eff /= eff.sum()  # 归一化（保持能量守恒）
    else:
        eff = psf

    tar = np.zeros((H, W), dtype=np.float32)
    if return_time_proj:
        t_num = np.zeros((H, W), dtype=np.float32)
        t_den = np.zeros((H, W), dtype=np.float32)

    for k in range(n_frames_i):
        t = k * float(frame_interval_s)
        ph = 2.0 * np.pi * (k / max(period_frames, 1e-6))

        perp = float(periodic_amp_px) * np.sin(ph + phase_perp)
        along = float(periodic_along_amp_px) * np.sin(0.6 * ph + phase_along)

        x = x0 + vx * t + dirx * along - diry * perp + rng.normal(0.0, jitter_px)
        y = y0 + vy * t + diry * along + dirx * perp + rng.normal(0.0, jitter_px)

        pk = float(peak) * (1.0 + 0.15 * np.sin(ph + phase_along) + rng.normal(0.0, scintillation))
        pk = max(0.0, pk)

        _place_kernel_add(tar, eff, y, x, pk)

        if return_time_proj:
            frame_id = (k + 1.0) / float(n_frames_i)
            _place_kernel_add(t_num, eff, y, x, pk * frame_id)
            _place_kernel_add(t_den, eff, y, x, pk)

    ratio = peak / (tar.max() + 1e-6)
    tar = ratio * tar
    out = tar + float(background)
    if shot_noise:
        out = np.clip(out, 0, None)
        out = rng.poisson(out).astype(np.float32)
    if read_noise > 0:
        out = out + rng.normal(0.0, float(read_noise), size=out.shape).astype(np.float32)

    if not return_time_proj:
        return out, tar

    t_proj = t_num / (t_den + 1e-6)
    t_proj = np.clip(t_proj, 0.0, 1.0).astype(np.float32)
    return out, tar, t_proj

def trunc_img(img, ratio=(0.5, 99.5)):
    vmin, vmax = np.percentile(img, ratio)
    img = np.clip(img, vmin, vmax)
    out = (img - vmin) / (vmax - vmin + 1e-8) * 255
    return out.astype(np.uint8)

def load_tj_img(path, denoise=True):
    img = cv2.imread(path, -1).astype(np.float32)
    if not denoise:
        return img
    med = cv2.medianBlur(img, 3)
    sub = img - med
    mask = (sub > med) & (med < med.mean()+med.std())
    oup = img.copy()
    oup[mask] = med[mask]
    return oup


## 静态背景真实数据

In [ ]:
a = 2048
path_dirs = ['/mnt/e/Imgs/03-TJStars/150ms-2k2k/decode/',
            '/mnt/e/Imgs/03-TJStars/3000ms-4k4k/decode/',
            '/mnt/e/Imgs/03-TJStars/TJ2/decode/']

### 导出测试数据（原始对齐图像）

In [ ]:
idx = 1
for path_dir in tqdm(path_dirs[:1], desc='Current Directory'):
    imgs = sorted(glob.glob(path_dir + '/*.tif'))
    if len(imgs) == 0:
        continue
    aligns = []
    for path in tqdm(imgs[1:], desc='Generate Mask'):
        img2 = load_tj_img(path)
        aligns.append(img2)
    aligns = np.array(aligns)
    oup = np.max(aligns, axis=0)
    max_idx = (np.argmax(aligns, axis=0) / len(imgs))*255

    h, w = oup.shape
    nh, nw = h // a, w // a
    for i in range(nh):
        for j in range(nw):
            y1, y2 = i * a, (i + 1) * a
            x1, x2 = j * a, (j + 1) * a
            roi = oup[y1:y2, x1:x2]
            t_roi = max_idx[y1:y2, x1:x2]
            vmin, vmax = np.percentile(roi, (0.5, 99.5))
            roi = (roi - vmin) / (vmax - vmin + 1e-8)
            roi = np.clip(roi, 0, 1)
            x8 = (roi * 255).astype(np.uint8)
            t8 = t_roi.astype(np.uint8)
            roi3 = np.stack([x8, t8, x8], axis=-1)
            if not cv2.imwrite(f'dataset/test_xt/still_{idx:03d}.png', roi3):
                print(f'Failed to save {idx}')
            idx += 1

### 导出测试数据（删除星点）

In [ ]:
idx = 1
for path_dir in tqdm(path_dirs[:], desc='Current Directory'):
    imgs = sorted(glob.glob(path_dir + '/*.tif'))
    if len(imgs) == 0: continue
    aligns = []
    for path in tqdm(imgs[1:], desc='Generate Mask'):
        img2 = load_tj_img(path)
        aligns.append(img2)
    aligns = np.array(aligns)
    oup = np.max(aligns, axis=0)
    med = np.median(aligns, axis=0)
    max_idx = (np.argmax(aligns, axis=0) / len(imgs))*255

    h, w = oup.shape
    nh, nw = h // a, w // a
    for i in range(nh):
        for j in range(nw):
            y1, y2 = i * a, (i + 1) * a
            x1, x2 = j * a, (j + 1) * a
            med_roi = med[y1:y2, x1:x2]
            bkg = sep.Background(np.ascontiguousarray(med_roi, np.float32))
            bkg_img = np.array(bkg)
            tars, ext_map = sep.extract(med_roi - bkg_img, 1.5, err=bkg.globalrms, deblend_cont=1, segmentation_map=True)
            star_mask_roi = ext_map > 0
            roi = oup[y1:y2, x1:x2]
            t_roi = max_idx[y1:y2, x1:x2]

            val = np.percentile(roi, 1)
            roi[star_mask_roi] = val
            vmin, vmax = np.percentile(roi, (0.01, 99.99))
            roi = (roi - vmin) / (vmax - vmin)
            roi = np.clip(roi, 0, 1)
            x8 = (roi * 255).astype(np.uint8)
            t8 = t_roi.astype(np.uint8)
            roi3 = np.stack([x8, t8, x8], axis=-1)
            if not cv2.imwrite(f'dataset/test_clean_xt/still_{idx:03d}.png', roi3):
                print(f'Failed to save {idx}')
            idx += 1

### 测试数据

In [ ]:
path_dirs = ['/mnt/e/Imgs/03-TJStars/150ms-2k2k/decode/',
            '/mnt/e/Imgs/03-TJStars/3000ms-4k4k/decode/',
            '/mnt/e/Imgs/03-TJStars/TJ2/decode/']

path_dir = path_dirs[0]
imgs = glob.glob(path_dir + '/*.tif')
aligns = []
for path in imgs:
    img2 = cv2.imread(path, -1)
    aligns.append(img2)
aligns = np.array(aligns)

In [ ]:
med = np.median(aligns, axis=0)
maxi = np.max(aligns, axis=0)
max_idx = np.argmax(aligns, axis=0)
bkg = sep.Background(np.ascontiguousarray(med, np.float32))
bkg_img = np.array(bkg)
tars, ext_map = sep.extract(med-bkg_img, 5, err=bkg.globalrms, deblend_cont=1, segmentation_map=True)
sub = maxi - med
bkg_val = bkg.globalback
sub[ext_map>0] = np.median(sub)

In [ ]:
mask = ext_map>0
bkg_sub = sep.Background(np.ascontiguousarray(sub, np.float32))
tars, fore = sep.extract(sub-np.array(bkg_sub), 1.5, err=bkg.globalrms, deblend_cont=1, segmentation_map=True)

map_ids = max_idx.copy()
map_ids[mask] = -1
map_ids[fore==0] = -1


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15,5))
vmin, vmax = np.percentile(sub, (0.5,99.5))
axes[0].imshow(sub, vmin=vmin, vmax=vmax, cmap='gray')
axes[1].imshow(fore>0, cmap='gray')
axes[2].imshow(map_ids, cmap='tab20')
for ax in axes: ax.axis('off')
plt.tight_layout()


## 动态背景真实数据

In [ ]:
a = 2048
path_dirs = ['/mnt/e/Imgs/04-RWZ/20220907/822070_2000/decode/',
            '/mnt/e/Imgs/03-TJStars/150ms-4k4k-motion/decode/',
            '/mnt/e/Imgs/03-TJStars/200ms-4k4k-motion/decode/',
            '/mnt/e/Imgs/03-TJStars/5000ms/decode/',
            '/mnt/e/Imgs/03-TJStars/TJ1/decode/',]


### 导出测试数据（原始对齐图像）

In [ ]:
idx = 1
for path_dir in tqdm(path_dirs[:1], desc='Current Directory'):
    imgs = sorted(glob.glob(path_dir + '/*.tif'))
    if len(imgs) == 0:
        continue
    img1 = load_tj_img(imgs[0])
    aligns = [img1]
    for path in tqdm(imgs[1:], desc='Generate Mask'):
        img2 = load_tj_img(path)
        img21, footprint = aa.register(img2, img1)
        aligns.append(img21)
    aligns = np.array(aligns)
    oup = np.max(aligns, axis=0)
    max_idx = (np.argmax(aligns, axis=0) / len(imgs))*255

    h, w = oup.shape
    nh, nw = h // a, w // a
    for i in range(nh):
        for j in range(nw):
            y1, y2 = i * a, (i + 1) * a
            x1, x2 = j * a, (j + 1) * a
            roi = oup[y1:y2, x1:x2]
            t_roi = max_idx[y1:y2, x1:x2]
            vmin, vmax = np.percentile(roi, (0.5, 99.5))
            roi = (roi - vmin) / (vmax - vmin + 1e-8)
            roi = np.clip(roi, 0, 1)
            x8 = (roi * 255).astype(np.uint8)
            t8 = t_roi.astype(np.uint8)
            roi3 = np.stack([x8, t8, x8], axis=-1)
            if not cv2.imwrite(f'dataset/test/{idx:03d}.png', roi3):
                print(f'Failed to save {idx}')
            idx += 1

### 导出测试数据(删除星点)

In [ ]:
idx = 1
for path_dir in tqdm(path_dirs[:], desc='Current Directory'):
    imgs = sorted(glob.glob(path_dir + '/*.tif'))
    if len(imgs) == 0:
        continue
    img1 = load_tj_img(imgs[0])
    aligns = [img1]
    valid_masks = [np.ones_like(img1, dtype=bool)]

    for path in tqdm(imgs[1:], desc='Generate Mask'):
        img2 = load_tj_img(path)
        img21, footprint = aa.register(img2, img1)
        aligns.append(img21)
        valid_masks.append(footprint)
    aligns = np.array(aligns)
    overlap_mask_all = np.logical_and.reduce(np.stack(valid_masks, axis=0), axis=0)
    oup = np.max(aligns, axis=0)
    med = np.median(aligns, axis=0)
    max_idx = (np.argmax(aligns, axis=0) / len(imgs))*255

    h, w = oup.shape
    nh, nw = h // a, w // a
    for i in range(nh):
        for j in range(nw):
            y1, y2 = i * a, (i + 1) * a
            x1, x2 = j * a, (j + 1) * a

            med_roi = med[y1:y2, x1:x2]
            bkg = sep.Background(np.ascontiguousarray(med_roi, np.float32))
            bkg_img = np.array(bkg)
            tars, ext_map = sep.extract(med_roi - bkg_img, 1.5, err=bkg.globalrms, deblend_cont=1, segmentation_map=True)
            star_mask_roi = ext_map > 0

            roi = oup[y1:y2, x1:x2]
            t_roi = max_idx[y1:y2, x1:x2]
            overlap_roi = overlap_mask_all[y1:y2, x1:x2]
            val = np.percentile(roi, 1)
            roi[star_mask_roi] = val
            roi[overlap_roi] = roi.min()
            vmin, vmax = np.percentile(roi, (0.01, 99.99))
            roi = (roi - vmin) / (vmax - vmin)
            roi = np.clip(roi, 0, 1)
            x8 = (roi * 255).astype(np.uint8)
            t8 = t_roi.astype(np.uint8)
            roi3 = np.stack([x8, t8, x8], axis=-1)
            if not cv2.imwrite(f'dataset/test_clean_xt/motion_{idx:03d}.png', roi3):
                print(f'Failed to save {idx}')
            idx += 1

### 测试数据

In [ ]:
path_dirs = ['/mnt/e/Imgs/04-RWZ/20220907/822070_2000/decode/',
            '/mnt/e/Imgs/03-TJStars/150ms-4k4k-motion/decode/',
            '/mnt/e/Imgs/03-TJStars/200ms-4k4k-motion/decode/',
            '/mnt/e/Imgs/03-TJStars/5000ms/decode/',
            '/mnt/e/Imgs/03-TJStars/TJ1/decode/',]

path_dir = path_dirs[-1]
imgs = glob.glob(path_dir + '/*.tif')
img1 = load_tj_img(imgs[0])
aligns = [img1]
for path in imgs[1:]:
    img2 = load_tj_img(path)
    img21, _ = aa.register(img2, img1)
    aligns.append(img21)
aligns = np.array(aligns)


In [ ]:
med = np.median(aligns, axis=0)
maxi = np.max(aligns, axis=0)
max_idx = np.argmax(aligns, axis=0)
bkg = sep.Background(np.ascontiguousarray(med, np.float32))
bkg_img = np.array(bkg)
tars, ext_map = sep.extract(med-bkg_img, 5, err=bkg.globalrms, deblend_cont=1, segmentation_map=True)
sub = maxi - med
bkg_val = bkg.globalback
sub[ext_map>0] = np.median(sub)


In [ ]:
mask = ext_map>0
bkg_sub = sep.Background(np.ascontiguousarray(sub, np.float32))
tars, fore = sep.extract(sub-np.array(bkg_sub), 1.5, err=bkg_sub.globalrms, deblend_cont=1, segmentation_map=True)

map_ids = max_idx.copy()


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15,5))
vmin, vmax = np.percentile(sub, (0.5,99.5))
axes[0].imshow(sub, vmin=vmin, vmax=vmax, cmap='gray')
axes[1].imshow(fore>0, cmap='gray')
axes[2].imshow(map_ids, cmap='tab20')
for ax in axes: ax.axis('off')
plt.tight_layout()


## OBB数据合成

### 相关函数

In [ ]:
def order_quad_clockwise(pts):
    pts = np.asarray(pts, dtype=np.float32).reshape(4, 2)
    c = pts.mean(axis=0)
    ang = np.arctan2(pts[:, 1] - c[1], pts[:, 0] - c[0])
    pts = pts[np.argsort(ang)]
    i0 = np.argmin(pts[:, 0] + pts[:, 1])
    pts = np.roll(pts, -i0, axis=0)
    return pts

def global_obb_from_projection_image(img: np.ndarray, thr: float, min_pixels: int = 4):
    """Build a stable OBB from thresholded target via geometric PCA after 5x5 dilation."""
    if img is None or img.size == 0 or not np.isfinite(thr):
        return None

    mask = (img > thr).astype(np.uint8)
    kernel = np.ones((3, 3), dtype=np.uint8)
    mask = cv2.dilate(mask, kernel, iterations=1)

    ys, xs = np.where(mask > 0)
    if len(xs) < min_pixels:
        return None

    pts = np.stack([xs, ys], axis=1).astype(np.float32)

    center = pts.mean(axis=0)
    pts0 = pts - center
    cov = np.cov(pts0, rowvar=False)
    if not np.all(np.isfinite(cov)):
        return None

    eigvals, eigvecs = np.linalg.eigh(cov)
    order = np.argsort(eigvals)[::-1]
    v1 = eigvecs[:, order[0]].astype(np.float32)
    v2 = eigvecs[:, order[1]].astype(np.float32)

    if np.cross(v1, v2) < 0:
        v2 = -v2

    p1 = pts0 @ v1
    p2 = pts0 @ v2
    min1, max1 = float(p1.min()), float(p1.max())
    min2, max2 = float(p2.min()), float(p2.max())

    if (max1 - min1) < 1e-6 or (max2 - min2) < 1e-6:
        return None

    local = np.array([
        [min1, min2],
        [max1, min2],
        [max1, max2],
        [min1, max2],
    ], dtype=np.float32)

    corners = center[None, :] + local[:, :1] * v1[None, :] + local[:, 1:] * v2[None, :]
    return order_quad_clockwise(corners)

def compose_frame_index_projection(t1, w1, t2, w2, rng, ratio=80):
    wsum = w1 + w2
    t_obj = (t1 * w1 + t2 * w2) / (wsum + 1e-6)

    bg = rng.uniform(0.0, 1.0, size=wsum.shape).astype(np.float32)
    if np.any(wsum > 0):
        nz = wsum[wsum > 0]
        thr = np.percentile(nz, 80.0)
        obj_mask = (wsum > max(thr, 1e-6)).astype(np.uint8)
        obj_mask = cv2.erode(obj_mask, np.ones((3, 3), np.uint8), iterations=2).astype(bool)
    else:
        obj_mask = np.zeros_like(wsum, dtype=bool)

    t = np.where(obj_mask, t_obj, bg)
    return np.clip(t, 0.0, 1.0).astype(np.float32)


def polygon_area(pts):
    x = pts[:, 0]
    y = pts[:, 1]
    return 0.5 * abs(np.dot(x, np.roll(y, -1)) - np.dot(y, np.roll(x, -1)))

def to_yolo_obb_line(obb, w, h):
    if obb[0] < 0:
        return None
    pts = np.asarray(obb, dtype=np.float32).reshape(4, 2)
    pts[:, 0] = np.clip(pts[:, 0], 0, w)
    pts[:, 1] = np.clip(pts[:, 1], 0, h)
    pts = order_quad_clockwise(pts)
    if polygon_area(pts) < 2.0:
        return None
    pts[:, 0] /= float(w)
    pts[:, 1] /= float(h)
    vals = np.clip(pts.reshape(-1), 0.0, 1.0)
    return '0 ' + ' '.join([f'{v:.6f}' for v in vals])

### 参数设置

In [ ]:
path_dirs = ['/mnt/e/Imgs/03-TJStars/150ms-2k2k/decode/',
            '/mnt/e/Imgs/03-TJStars/3000ms-4k4k/decode/',
            '/mnt/e/Imgs/03-TJStars/TJ2/decode/']

root_dir = 'dataset/obb_clean_xt'
out_root = Path('dataset/yolo_obb_clean_xt')
imgs_dir_x = Path(f'{root_dir}/imgs_x')
imgs_dir_t = Path(f'{root_dir}/imgs_t')
label_file = Path(f'{root_dir}/labels_obb.txt')
patch_size = 1024
stride = 512
start_idx = 1
FLAG_BKG = False    # if destar

train_ratio = 0.8
seed = 42
class_names = ['line']
clean_output = True
RATIO_X = (0.01, 99.99) if FLAG_BKG else (0.5, 99.5) 

### 生成空域-时序投影 仿图

In [ ]:
imgs_dir_x.mkdir(parents=True, exist_ok=True)
imgs_dir_t.mkdir(parents=True, exist_ok=True)
idx = start_idx
content = []

meds = []
for path_dir in path_dirs:
    aligns = []
    paths = glob.glob(path_dir + '*.tif')
    for path in tqdm(paths, desc='Generate Aigned Img'):
        aligns.append(cv2.imread(path, -1))
    if FLAG_BKG:
        med = np.median(np.array(aligns), axis=0)
        bkg = sep.Background(np.ascontiguousarray(med, np.float32))
        bkg_img = np.array(bkg)
        tars, ext_map = sep.extract(med - bkg_img, 5, err=bkg.globalrms, deblend_cont=1, segmentation_map=True)
        meds.append(ext_map)

for num_dir, path_dir in enumerate(tqdm(path_dirs, desc='Simulating Img')):
    if FLAG_BKG: mask = meds[num_dir] > 0
    paths = glob.glob(path_dir + '*.tif')
    for path in paths:
        ori = cv2.imread(path, -1)
        if ori is None:
            continue
        ori = ori.astype(np.float32)
        h, w = ori.shape
        nw = (w - patch_size) // stride + 1
        nh = (h - patch_size) // stride + 1

        for i in range(nh):
            for j in range(nw):
                x1, y1 = j * stride, i * stride
                x2, y2 = x1 + patch_size, y1 + patch_size
                img = ori[y1:y2, x1:x2]
                if FLAG_BKG:
                    mask_roi = mask[y1:y2, x1:x2]
                    img[mask_roi] = np.median(img)

                try:
                    seed = idx
                    rng = np.random.default_rng(seed)
                    exp = max(rng.normal(1, 0.3), 0.05)

                    n_pt = int(rng.integers(5, 30))
                    peak = max(rng.normal(0.1, 0.03), 0.001) * img.max() if FLAG_BKG else max(rng.normal(1, 0.5), 0.1) * img.std()
                    img_pt, tar_pt, t_pt = simulate_trajectory_projection_with_interval(
                        seed=seed,
                        img_shape=img.shape,
                        n_frames=n_pt,
                        speed_px_s=rng.integers(1, 5),
                        exposure_s=exp,
                        frame_interval_s=exp * rng.integers(2, 15),
                        radius_px=max(rng.normal(2, 0.7), 0.5),
                        peak=peak,
                        angle_deg=rng.uniform(-90, 90),
                        background=0,
                        return_time_proj=True,
                    )

                    seed += 1
                    rng = np.random.default_rng(seed)
                    n_line = int(rng.integers(5, 30))
                    peak = max(rng.normal(0.1, 0.03), 0.001) * img.max() if FLAG_BKG else max(rng.normal(0.5, 0.5), 0.1) * img.std()
                    img_line, tar_line, t_line = simulate_trajectory_projection_with_interval(
                        seed=seed,
                        img_shape=img.shape,
                        n_frames=n_line,
                        speed_px_s=rng.integers(3, 23),
                        exposure_s=exp,
                        frame_interval_s=exp * rng.integers(2, 5),
                        radius_px=max(rng.normal(1, 0.3), 0.2),
                        peak=peak,
                        angle_deg=rng.uniform(-90, 90),
                        background=0,
                        return_time_proj=True,
                    )

                    syn_x = (img + img_pt + img_line).astype(np.float32)
                    ratio = 80 if FLAG_BKG else 50
                    syn_t = compose_frame_index_projection(t_pt, tar_pt, t_line, tar_line, np.random.default_rng(idx + 9999), )

                    row = [idx]
                    for tar in [tar_line, tar_pt]:
                        th = np.percentile(tar, 99.9)
                        obb = global_obb_from_projection_image(tar, th)
                        if obb is None:
                            row += [-1.0] * 8
                        else:
                            row += obb.reshape(-1).tolist()
                    content.append(row)

                    cv2.imwrite(str(imgs_dir_x / f'{idx:03d}.tif'), syn_x)
                    cv2.imwrite(str(imgs_dir_t / f'{idx:03d}.tif'), (syn_t * 65535.0).astype(np.uint16))
                    idx += 1
                except Exception as e:
                    print(f'Failed idx={idx}: {e}')

with label_file.open('w', encoding='utf-8') as f:
    for row in content:
        vals = ','.join([f'{row[0]:03d}'] + [f'{v:.6f}' for v in row[1:]])
        f.write(vals + '\n')

print(f'\n\nFLAG_BKG: {FLAG_BKG}. \nOBB sim done. images={len(content)}, labels={label_file}, X={imgs_dir_x}, T={imgs_dir_t}')


### 转为yolo-obb格式

In [ ]:
def write_split(pairs, split):
    for img_x_path, t_path, (obb1, obb2) in tqdm(pairs, desc="writing..."):
        img_x = cv2.imread(str(img_x_path), -1)
        img_t = cv2.imread(str(t_path), -1)
        if img_x is None:
            raise ValueError(f'Failed to read X image: {img_x_path}')
        if img_t is None:
            raise ValueError(f'Failed to read T image: {t_path}')

        h, w = img_x.shape[:2]
        img_x8 = trunc_img(img_x, RATIO_X)

        img_t = img_t.astype(np.float32)
        if img_t.max() > 1.5:
            img_t = img_t / 65535.0
        img_t8 = np.clip(img_t, 0.0, 1.0) * 255.0
        img_t8 = img_t8.astype(np.uint8)

        img3 = np.stack([img_x8, img_t8, img_x8], axis=-1)
        dst_img = out_root / 'images' / split / f'{img_x_path.stem}.png'
        cv2.imwrite(str(dst_img), img3)

        lines = []
        for obb in (obb1, obb2):
            line = to_yolo_obb_line(obb, w=w, h=h)
            if line is not None:
                lines.append(line)

        dedup = []
        seen = set()
        for line in lines:
            if line not in seen:
                dedup.append(line)
                seen.add(line)

        dst_lbl = out_root / 'labels' / split / f'{img_x_path.stem}.txt'
        with dst_lbl.open('w', encoding='utf-8') as f:
            if dedup:
                f.write('\n'.join(dedup) + '\n')



assert imgs_dir_x.exists(), f'Missing image dir: {imgs_dir_x}'
assert imgs_dir_t.exists(), f'Missing frame-index dir: {imgs_dir_t}'
assert label_file.exists(), f'Missing labels file: {label_file}'

if clean_output and out_root.exists():
    shutil.rmtree(out_root)

exts = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff', '.webp'}
images_x = sorted([p for p in imgs_dir_x.iterdir() if p.is_file() and p.suffix.lower() in exts])
assert images_x, f'No images found in {imgs_dir_x}'

labels_by_idx = {}
with label_file.open('r', encoding='utf-8') as f:
    for ln, line in enumerate(f, 1):
        line = line.strip()
        if not line:
            continue
        parts = [x.strip() for x in line.split(',')]
        if len(parts) != 17:
            raise ValueError(f'Bad OBB label format at line {ln}: expect 17 fields, got {len(parts)}')
        idx = int(parts[0])
        vals = list(map(float, parts[1:]))
        labels_by_idx[idx] = (vals[:8], vals[8:16])

paired = []
for img_x_path in images_x:
    img_idx = int(img_x_path.stem)
    t_path = imgs_dir_t / f'{img_x_path.stem}{img_x_path.suffix}'
    if img_idx not in labels_by_idx:
        raise ValueError(f'Missing label for image: {img_x_path.name}')
    if not t_path.exists():
        raise ValueError(f'Missing T map for image: {img_x_path.name}')
    paired.append((img_x_path, t_path, labels_by_idx[img_idx]))

for split in ['train', 'val']:
    (out_root / 'images' / split).mkdir(parents=True, exist_ok=True)
    (out_root / 'labels' / split).mkdir(parents=True, exist_ok=True)

rng = random.Random(seed)
rng.shuffle(paired)
train_count = int(len(paired) * train_ratio)
train_set = paired[:train_count]
val_set = paired[train_count:]

write_split(train_set, 'train')
write_split(val_set, 'val')

yaml_text = (
    f'path: {out_root.as_posix()}\n'
    'train: images/train\n'
    'val: images/val\n'
    'names:\n'
    f'  0: {class_names[0]}\n'
)
(out_root / 'data.yaml').write_text(yaml_text, encoding='utf-8')

print(f'Total paired images: {len(paired)}')
print(f'Train: {len(train_set)} | Val: {len(val_set)}')
print(f'YOLO-OBB dataset ready at: {out_root}')

### 抽样可视化（X / T / OBB）
随机抽样查看合成结果：左图为X（抑星投影），中图为T（帧号投影），右图为X上叠加OBB标签。


In [ ]:
def _draw_obb(ax, pts, color='lime', lw=1.5):
    pts = np.asarray(pts, dtype=np.float32).reshape(4, 2)
    poly = np.vstack([pts, pts[:1]])
    ax.plot(poly[:, 0], poly[:, 1], color=color, linewidth=lw)

def _read_yolo_obb_label(label_path, w, h):
    obbs = []
    if not label_path.exists():
        return obbs
    with open(label_path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            parts = line.split()
            if len(parts) != 9:
                continue
            vals = np.array(list(map(float, parts[1:])), dtype=np.float32).reshape(4, 2)
            vals[:, 0] *= float(w)
            vals[:, 1] *= float(h)
            obbs.append(vals)
    return obbs

def _crop_from_obb(obb, h, w, pad=20, scale=2.0):
    x0, y0 = obb[:, 0].min(), obb[:, 1].min()
    x1, y1 = obb[:, 0].max(), obb[:, 1].max()
    cx, cy = 0.5 * (x0 + x1), 0.5 * (y0 + y1)
    bw, bh = max(2.0, x1 - x0), max(2.0, y1 - y0)
    half = 0.5 * max(bw, bh) * scale + pad
    xa = int(max(0, np.floor(cx - half)))
    xb = int(min(w, np.ceil(cx + half)))
    ya = int(max(0, np.floor(cy - half)))
    yb = int(min(h, np.ceil(cy + half)))
    return xa, ya, xb, yb

def sample_visualize_yolo_obb_dataset(
    dataset_root='dataset/yolo_obb_auto_xt',
    split='train',
    n_samples=6,
    seed=0,
    zoom_pad=20,
    zoom_scale=2.0,
):
    dataset_root = Path(dataset_root)
    img_dir = dataset_root / 'images' / split
    lbl_dir = dataset_root / 'labels' / split
    exts = {'.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff', '.webp'}
    imgs = sorted([p for p in img_dir.iterdir() if p.is_file() and p.suffix.lower() in exts])
    if not imgs:
        raise ValueError(f'No images found in {img_dir}')

    rng = random.Random(seed)
    samples = rng.sample(imgs, k=min(n_samples, len(imgs)))

    for p in samples:
        img = cv2.imread(str(p), cv2.IMREAD_COLOR)
        if img is None:
            print(f'[skip] read failed: {p.name}')
            continue
        h, w = img.shape[:2]
        x8 = img[:, :, 0]
        t8 = img[:, :, 1]
        obbs = _read_yolo_obb_label(lbl_dir / f'{p.stem}.txt', w, h)

        n_zoom = max(1, len(obbs))
        cols = 3 + n_zoom
        fig, axes = plt.subplots(1, cols, figsize=(4.0 * cols, 4.2))

        axes[0].imshow(x8, cmap='gray')
        axes[0].set_title(f'X | {split}/{p.name}')
        axes[1].imshow(t8, cmap='turbo', vmin=0, vmax=255)
        axes[1].set_title('T')
        axes[2].imshow(x8, cmap='gray')
        axes[2].set_title(f'X + OBB ({len(obbs)})')

        crop_boxes = []
        for obb in obbs:
            _draw_obb(axes[2], obb, color='lime', lw=1.8)
            xa, ya, xb, yb = _crop_from_obb(obb, h, w, pad=zoom_pad, scale=zoom_scale)
            crop_boxes.append((xa, ya, xb, yb))
            axes[2].add_patch(plt.Rectangle((xa, ya), xb - xa, yb - ya, fill=False, edgecolor='yellow', linewidth=1.2))

        if not obbs:
            axes[3].imshow(x8, cmap='gray')
            axes[3].set_title('No OBB label')
        else:
            for k, obb in enumerate(obbs):
                xa, ya, xb, yb = crop_boxes[k]
                ax = axes[3 + k]
                crop_x = x8[ya:yb, xa:xb]
                crop_t = t8[ya:yb, xa:xb]
                show = crop_t
                # show = np.stack([crop_x, crop_t, crop_x], axis=-1)
                ax.imshow(show)
                obb_local = obb.copy()
                obb_local[:, 0] -= xa
                obb_local[:, 1] -= ya
                _draw_obb(ax, obb_local, color='red', lw=2.0)
                ax.set_title(f'Zoom target {k + 1}')

        for ax in axes:
            ax.axis('off')
        plt.tight_layout()
        plt.show()

sample_visualize_yolo_obb_dataset(dataset_root=out_root, split='train', n_samples=6, seed=0)

## 模型测试

单张图片

In [ ]:
model = YOLO("runs/obb/yolo11n_obb_xt_v1/weights/best.pt")  # pretrained YOLO26n model

results = model(["dataset/test_clean_xt/still_001.png"], imgsz=2048, conf=0.25)  # return a list of Results objects

for result in results:
    boxes = result.boxes  # Boxes object for bounding box outputs
    masks = result.masks  # Masks object for segmentation masks outputs
    keypoints = result.keypoints  # Keypoints object for pose outputs
    probs = result.probs  # Probs object for classification outputs
    obb = result.obb  # Oriented boxes object for OBB outputs
    result.show()  # display to screen
    result.save(filename="result.jpg")  # save to disk


### 测试原图

In [ ]:
direc = 'runs/obb/yolo11n_obb_xt_v1'
test_dir = 'dataset/test_xt'
conf = 0.25
model = YOLO(f"{direc}/weights/best.pt")

img_paths = sorted(glob.glob(f'{test_dir}/still*.png'))
test_files = [cv2.imread(x) for x in img_paths]
test_files = [x for x in test_files if x is not None]
results = model(test_files, imgsz=test_files[0].shape[0], conf=conf)

n = len(results)
n_rows, n_cols = int(n/2+ 0.5), 2
fig, axes = plt.subplots(n_rows, n_cols, figsize=(6 * n_cols, 6 * n_rows))
axes = [axes] if (n_rows == 1 and n_cols == 1) else axes.ravel()
for i, result in enumerate(results):
    im = result.plot(labels=False)
    axes[i].imshow(im[:, :, ::-1])
    axes[i].axis('off')
for j in range(n, len(axes)): axes[j].axis('off')
plt.tight_layout()
plt.show()

### 测试星点抑制图

In [ ]:
direc = 'runs/obb/yolo11n_obb_xt_v1'
test_dir = 'dataset/test_clean_xt'
conf = 0.01
model = YOLO(f"{direc}/weights/last.pt")

img_paths = sorted(glob.glob(f'{test_dir}/still*.png'))
test_files = [cv2.imread(x) for x in img_paths]
test_files = [x for x in test_files if x is not None]
results = model(test_files, imgsz=test_files[0].shape[0], conf=conf)

n = len(results)
n_rows, n_cols = int(n/2+ 0.5), 2
fig, axes = plt.subplots(n_rows, n_cols, figsize=(6 * n_cols, 6 * n_rows))
axes = [axes] if (n_rows == 1 and n_cols == 1) else axes.ravel()
for i, result in enumerate(results):
    im = result.plot(labels=False)
    axes[i].imshow(im[:, :, ::-1])
    axes[i].axis('off')
for j in range(n, len(axes)): axes[j].axis('off')
plt.tight_layout()
plt.show()
